# **Metode Eliminasi Gauss**

Eliminasi Gauss merupakan salah satu metode fundamental untuk menyelesaikan SPL.

Misalkan:

$$\begin{cases} 2x + y = 5 \\ x - y = 1 \end{cases}$$

Kita ubah menjadi matriks augmented:

$$\left[ \begin{array}{cc|c} 2 & 1 & 5 \\ 1 & -1 & 1 \end{array} \right]$$

Tujuannya adalah mengubah elemen di bawah diagonal utama menjadi nol.

Secara umum:

$$\begin{bmatrix} a_{11} & a_{12} & a_{13} \\ a_{21} & a_{22} & a_{23} \\ a_{31} & a_{32} & a_{33} \end{bmatrix}$$

diubah menjadi bentuk:

$$\begin{bmatrix} a_{11} & a_{12} & a_{13} \\ 0 & a'_{22} & a'_{23} \\ 0 & 0 & a''_{33} \end{bmatrix}$$

Setelah itu kita melakukan **back substitution**.

In [3]:
import numpy as np

def eliminasi_gauss(A, b):
    A = A.astype(float)
    b = b.astype(float)

    n = len(b)

    # Membentuk augmented matrix
    M = np.column_stack((A, b))

    # Forward elimination
    for i in range(n):
        for j in range(i + 1, n):
            faktor = M[j, i] / M[i, i]
            M[j] = M[j] - faktor * M[i]
    
    # Back substitution
    x = np.zeros(n)

    for i in range(n - 1, -1, -1):
        jumlah = np.dot(M[i, i + 1:n], x[i + 1:n])
        x[i] = (M[i, -1] - jumlah) / M[i, i]
    
    return x

In [4]:
A = np.array([
    [2, 3],
    [1,-1]
])

b = np.array([8, 1])

x = eliminasi_gauss(A, b)

print(x)

[2.2 1.2]


### **Masalah Pivot**

Implementasi sebelumnya memiliki keleahan.

Bagaimana jika:

$$ a_{ii} = 0? $$

Misalnya:

$$\begin{bmatrix} 0 & 2 \\ 1 & 3 \end{bmatrix}$$

Kita tidak dapat melakukan:

$$\frac{a_{21}}{a_{11}}$$

karena:

$$a_{11} = 0$$

Solusinya adalah **pertukaran baris**.

Ini disebut **pivoting**.

### **Partial Pivoting**

Pada partial pivoting, kita mencari elemen dengan nilai absolut terbesar pada kolom yang sedang digunakan sebagai pivot.

Contoh:

$$\begin{bmatrix} 0.001 & 2 \\ 1 & 3 \end{bmatrix}$$

daripada menggunakan $0.001$ sebagai pivot, lebih baik menukar baris sehingga:

$$\begin{bmatrix} 1 & 3 \\ 0.001 & 2 \end{bmatrix}$$

Hal ini meningkatkan kestabilan numerik.

Implementasinya:

In [15]:
def eliminasi_gauss_pivot(A, b):
    A = A.astype(float)
    b = b.astype(float)

    n = len(b)

    M = np.column_stack((A, b))

    for i in range(n):
        # Mencari pivot terbesar
        pivot = np.argmax(np.abs(M[i:, i])) + i

        # Tukar baris
        M[[i ,pivot]] = M[[pivot, i]]

        for j in range(i + 1, n):
            faktor = M[j, i] / M[i, i]
            M[j] -= faktor * M[i]

    # back substition
    x = np.zeros(n)

    for i in range(n - 1, -1, -1):
        jumlah = np.dot(M[i, i + 1:n], x[i + 1:n])
        x[i] = (M[i, -1] - jumlah) / M[i, i]

    return x

In [19]:
A = np.array([
    [2, 3],
    [1,-1]
])

b = np.array([8, 1])

x = eliminasi_gauss_pivot(A, b)

print(x)

[2.2 1.2]


# **Metode Gauss-Jordan**

Eliminasi Gauss menghasilkan matriks segitiga atas.

Gauss-Jordan melangkah lebih jauh sampai menghasilkan:

$$\left[ \begin{array}{cc|c} 1 & 0 & x \\ 0 & 1 & y \end{array} \right]$$

Misalnya:

$$\left[ \begin{array}{cc|c} 2 & 3 & 8 \\ 1 & -1 & 1 \end{array} \right]$$

diubah melalui operasi baris menjadi:

$$\left[ \begin{array}{cc|c} 1 & 0 & 2.2 \\ 0 & 1 & 1.2 \end{array} \right]$$

Sehingga solusi langsung terlihat.

In [21]:
import numpy as np

def gauss_jordan(A, b):
    A = A.astype(float)
    b = b.astype(float)

    n = len(b)

    M = np.column_stack((A, b))

    for i in range(n):
        # Pivoting
        pivot = np.argmax(np.abs(M[i:, i])) + i
        M[[i, pivot]] = M[[pivot, i]]

        # Membuat pivot = 1
        M[i] = M[i] / M[i,i]

        # Eliminasi semua baris lainnya
        for j in range(n):
            if j != i:
                faktor = M[j,i]
                M[j] -= faktor * M[i]

    return M[:, -1]

In [22]:
A = np.array([
    [2, 3],
    [1,-1]
])

b = np.array([8, 1])

x = gauss_jordan(A, b)

print(x)

[2.2 1.2]


# **Metode Faktorisasi LU**

Metode LU memecah matriks:

$$A$$

menjadi:

$$\boxed{A = LU}$$

dengan:
* $L =$ Lower triangular matrix
* $U =$ Upper triangular matrix

Sehingga:

$$\mathbf{A x = b}$$

menjadi:

$$LU \mathbf{x = b}$$

Kita definisikan:

$$L \mathbf{y = b}$$

kemudian:

$$U \mathbf{x = y}$$

Jadi terdapat dua tahap:
1. Forward substitution
2. Back substitution

LU sangat berguna jika kita harus menyelesaikan banyak SPL dengan matriks $A$ yang sama tetapi $\mathbf{b}$ berbeda.

In [23]:
import numpy as np
from scipy.linalg import lu

A = np.array([
    [2, 3],
    [1, -1]
], dtype=float)

P, L, U = lu(A)

print("P =")
print(P)

print("L =")
print(L)

print("U =")
print(U)

P =
[[1. 0.]
 [0. 1.]]
L =
[[1.  0. ]
 [0.5 1. ]]
U =
[[ 2.   3. ]
 [ 0.  -2.5]]


# **Metode Jacobi**

Metode sebelumnya termasuk **metode langsung**.

Searang kita masuk ke **metode interatif**.

Misalkan:

$$10x + y = 11$$
$$x + 10y = 11$$

Kita isolasi:

$$x = \frac{11 - y}{10}$$

dan

$$y = \frac{11 - x}{10}$$

Kita mulai dengan tebakan awal:

$$x^{(0)} = 0$$
$$y^{(0)} = 0$$

Kemudian:

$$x^{(1)} = \frac{11 - y^{(0)}}{10} = 1.1$$

$$y^{(1)} = \frac{11 - x^{(0)}}{10} = 1.1$$

Iterasi berikutnya:

$$x^{(2)} = \frac{11 - 1.1}{10} = 0.99$$

dan seterusnya.

Nilai tersebut akhirnya mendekati:

$$x = 1, \quad y = 1$$

In [2]:
import numpy as np

def jacobi(A, b, x0=None, tol=1e-10, max_iter=100):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)

    n = len(b)

    if x0 is None:
        x = np.zeros(n)
    else:
        x = np.array(x0, dtype=float)

    for k in range(max_iter):
        x_new = np.zeros(n)

        for i in range(n):
            jumlah = 0

            for j in range(n):
                if j != i:
                    jumlah += A[i, j] * x[j]

            x_new[i] = (b[i] - jumlah) / A[i,i]

        error = np.linalg.norm(x_new - x)

        if error < tol:
            return x_new, k + 1

        x = x_new

    return x, max_iter

In [3]:
A = np.array([
    [10, 1],
    [1, 10]
])

b = np.array([11,11])

solusi, iterasi = jacobi(A,b)

print(f"Solusi  : {solusi}")
print(f"Iterasi : {iterasi}")

Solusi  : [1. 1.]
Iterasi : 12


# **Metode Gauss-Seidel**

Gauss-Seidel mirip dengan Jacobi.

Perbedaannya sangat penting:

**Jacobi menggunakan nilai dari iterasi sebelumnya untuk semua variabel.**

Sedangkan Gauss-Seidel langsung menggunakan nilai terbaru yang sudah dihitung.

Misalnya:

$$x^{(k+1)}$$

sudah diperoleh.

Gauss-Seidel langsung menggunakan nilai tersebut ketika menghitung:

$$y^{(k+1)}$$

Akibatnya, Gauss-Seidel sering konvergen lebih cepat daripada Jacobi.

In [4]:
import numpy as np

def gauss_seidel(A, b, x0=None, tol=1e-10, max_iter=100):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)

    n = len(b)

    if x0 is None:
        x = np.zeros(n)
    else:
        x = np.array(x0, dtype=float)

    for k in range(max_iter):
        x_old = x.copy()

        for i in range(n):
            jumlah = 0

            for j in range(n):
                if j != i:
                    jumlah += A[i,j] * x[j]

            x[i] = (b[i] - jumlah) / A[i,i]

        error = np.linalg.norm(x - x_old)

        if error < tol:
            return x, k + 1

    return x, max_iter

In [5]:
A = np.array([
    [10, 1],
    [1, 10]
])

b = np.array([11,11])

solusi, iterasi = gauss_seidel(A,b)

print(f"Solusi  : {solusi}")
print(f"Iterasi : {iterasi}")

Solusi  : [1. 1.]
Iterasi : 7


# **Penyelesaian SPL dengan NumPy**

In [6]:
import numpy as np

A = np.array([
    [2, 3],
    [1,-1]
])

b = np.array([8, 1])

x = np.linalg.solve(A,b)

print(x)

[2.2 1.2]


Misalkan kita memiliki rangkaian listrik dengan beberapa node. Setelah menerapkan hukum Kirchhoff, kita dapat memperoleh sistem:

$$3V_1 - V_2 = 10$$
$$-V_1 + 4V_2 - V_3 = 0$$
$$-V_2 + 3V_3 = 5$$

Dalam bentuk matriks:

$$\begin{bmatrix} 3 & -1 & 0 \\ -1 & 4 & -1 \\ 0 & -1 & 3 \end{bmatrix} \begin{bmatrix} V_1 \\ V_2 \\ V_3 \end{bmatrix} = \begin{bmatrix} 10 \\ 0 \\ 5 \end{bmatrix}$$

Kita dapat menyelesaikannya menggunakan:

In [8]:
import numpy as np

A = np.array([
    [3, -1, 0],
    [-1, 4, -1],
    [0, -1, 3]
])

b = np.array([10, 0, 5], dtype=float)

V = np.linalg.solve(A, b)

print(f"V1 = {V[0]}")
print(f"V2 = {V[1]}")
print(f"V3 = {V[2]}")

V1 = 3.8333333333333335
V2 = 1.5
V3 = 2.166666666666667


In [9]:
r = b - A @ V

print(f"Residual      = {r}")
print(f"Norm residual = {np.linalg.norm(r)}")

Residual      = [ 0.0000000e+00  4.4408921e-16 -8.8817842e-16]
Norm residual = 9.930136612989092e-16
